# Explosive Breaching Demo with Qwen2.5-VL

Modular implementation of the explosive breaching demo.
- **VLM Client**: Connects to Qwen2.5-VL served via vLLM
- **ROS2 Node**: Subscribes to behavior status, publishes behavior commands
- **Behavior Builders**: Modular functions to construct each behavior command
- **Coordinator**: Sequences behaviors and verifies completion

## 1. Imports

In [1]:
import json
import base64
from pathlib import Path

import numpy as np
import rclpy
from rclpy.node import Node
from rclpy.qos import QoSProfile, QoSReliabilityPolicy, QoSHistoryPolicy
from behavior_msgs.msg import (
    AI2RCommandMessage,
    AI2RStatusMessage,
    AI2RNavigationMessage,
    AI2RReceiveObjectMessage,
)
from openai import OpenAI

print("Imports loaded.")

Imports loaded.


## 2. VLM Client Setup

In [ ]:
import io
from PIL import Image

vlm_client = OpenAI(base_url="http://localhost:8000/v1", api_key="unused")
VLM_MODEL = vlm_client.models.list().data[0].id
print(f"Connected to VLM: {VLM_MODEL}")


def vlm_text_query(prompt, system_prompt="You are a helpful assistant.", max_tokens=256):
    """Send a text-only query to the VLM."""
    response = vlm_client.chat.completions.create(
        model=VLM_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        max_tokens=max_tokens,
        temperature=0.3,
    )
    return response.choices[0].message.content


def vlm_image_query(image_path, prompt, max_tokens=256, max_size=448):
    """Send an image + text query to the VLM.

    Images are resized to fit within max_size x max_size (preserving aspect ratio)
    before encoding to keep the token count within the server's max_model_len.
    Qwen2.5-VL uses 14x14 patches, so 448px → ~1024 visual tokens.
    """
    img = Image.open(image_path).convert("RGB")
    img.thumbnail((max_size, max_size), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=90)
    img_b64 = base64.b64encode(buf.getvalue()).decode()

    response = vlm_client.chat.completions.create(
        model=VLM_MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}},
                {"type": "text", "text": prompt},
            ],
        }],
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


## 3. Behavior Builders

Each function builds an `AI2RCommandMessage` for a specific behavior type.

In [3]:
def build_scan_command():
    """Build a SCAN behavior command."""
    cmd = AI2RCommandMessage()
    cmd.behavior_to_execute = "SCAN"
    cmd.adapting_behavior = False
    return cmd


def build_goto_command(target_object, spatial_relation=AI2RNavigationMessage.DEFAULT,
                       pov_object="", distance=1.0):
    """Build a GOTO behavior command."""
    cmd = AI2RCommandMessage()
    cmd.behavior_to_execute = "GOTO"
    cmd.adapting_behavior = True

    nav = AI2RNavigationMessage()
    nav.target_object = target_object
    nav.distance_to_object = distance
    nav.spatial_relation = spatial_relation
    nav.pov_object = pov_object
    if spatial_relation == AI2RNavigationMessage.DEFAULT or pov_object == "":
        nav.pov_object = "walkingFrame"

    cmd.navigation = nav
    return cmd


def build_receive_object_command(object_name, side=1):
    """Build a RECEIVE OBJECT behavior command."""
    cmd = AI2RCommandMessage()
    cmd.behavior_to_execute = "RECEIVE OBJECT"
    cmd.adapting_behavior = True

    receive = AI2RReceiveObjectMessage()
    receive.object_name = object_name
    receive.side = bytes([side])

    cmd.receive_object = receive
    return cmd


def build_place_charge_command():
    """Build a PLACE CHARGE ON DOOR behavior command."""
    cmd = AI2RCommandMessage()
    cmd.behavior_to_execute = "PLACE CHARGE ON DOOR"
    cmd.adapting_behavior = False
    return cmd


print("Behavior builders defined.")

Behavior builders defined.


## 4. Scene Parser

Extracts scene objects and available behaviors from the status message.

In [4]:
def parse_scene(msg):
    """Parse objects and available behaviors from a status message."""
    objects = []
    if msg.objects:
        for obj in msg.objects:
            objects.append({
                "name": obj.object_name,
                "pose_in_world": obj.object_pose_in_world,
                "pose_in_robot_frame": obj.object_pose_in_robot_frame,
            })

    behaviors = list(msg.available_behaviors) if msg.available_behaviors else []
    return objects, behaviors


def log_failure(msg):
    """Extract and log failure info from a status message. Returns failure dict or None."""
    if msg.failed_behavior == "-":
        return None

    failure = msg.failure
    failure_info = {
        "Failed behavior": msg.failed_behavior,
        "Description": failure.action_name,
        "Type": failure.action_type,
    }

    if failure.missing_frame:
        failure_info["Missing Frame"] = failure.reference_frame
    if failure.collision_name != "-":
        failure_info["Collision with"] = failure.collision_name

    position_error = failure.position_error
    error_vector = np.array([position_error.x, position_error.y, position_error.z])
    norm = np.linalg.norm(error_vector)
    if norm > failure.position_tolerance:
        failure_info["Position error"] = float(norm)

    with open("failure_info.json", "a") as f:
        json.dump(failure_info, f, indent=4)

    return failure_info


print("Scene parser and failure logger defined.")

Scene parser and failure logger defined.


## 5. Mission Definition

Define the sequence of behavior commands for the explosive breaching mission.

In [5]:
MISSION_COMMANDS = [
    build_scan_command(),
    build_goto_command("Person1", AI2RNavigationMessage.DEFAULT),
    build_receive_object_command("Charge1"),
    build_goto_command("DoorPanel1", AI2RNavigationMessage.FRONT),
    build_place_charge_command(),
    build_goto_command("Barrier1", AI2RNavigationMessage.BEHIND),
]

print(f"Mission defined with {len(MISSION_COMMANDS)} behaviors:")
for i, cmd in enumerate(MISSION_COMMANDS):
    print(f"  {i + 1}. {cmd.behavior_to_execute}")

Mission defined with 6 behaviors:
  1. SCAN
  2. GOTO
  3. RECEIVE OBJECT
  4. GOTO
  5. PLACE CHARGE ON DOOR
  6. GOTO


## 6. Behavior Coordinator Node

ROS2 node that sequences through the mission, verifying each behavior completes before sending the next.

In [6]:
class BehaviorCoordinator(Node):
    def __init__(self, mission_commands):
        super().__init__("behavior_coordination_node")

        self.mission_commands = mission_commands
        self.command_index = 0
        self.initialized = False
        self.logged_failure = False
        self.last_commanded_behavior = None
        self.last_printed_completion = None
        self.demo_complete = False

        qos_best_effort = QoSProfile(
            reliability=QoSReliabilityPolicy.BEST_EFFORT,
            history=QoSHistoryPolicy.KEEP_LAST,
            depth=1,
        )
        qos_reliable = QoSProfile(
            reliability=QoSReliabilityPolicy.RELIABLE,
            history=QoSHistoryPolicy.KEEP_LAST,
            depth=1,
        )

        self.status_sub = self.create_subscription(
            AI2RStatusMessage,
            "/ihmc/behavior_tree/ai2r_status",
            self.on_status,
            qos_best_effort,
        )
        self.command_pub = self.create_publisher(
            AI2RCommandMessage,
            "/ihmc/behavior_tree/ai2r_command",
            qos_reliable,
        )

    def on_status(self, msg):
        if self.demo_complete:
            return

        # Print scene info on first message
        if not self.initialized:
            objects, behaviors = parse_scene(msg)
            print("Objects in the scene:")
            for obj in objects:
                print(f"  {obj['name']}")
            print("Available behaviors:")
            for b in behaviors:
                print(f"  {b}")

        # Monitor completion
        if (msg.completed_behavior != "-"
                and msg.behavior_in_progress == "-"
                and msg.completed_behavior != self.last_printed_completion):
            print(f"Completed: {msg.completed_behavior}")
            self.last_printed_completion = msg.completed_behavior

        # Log failures
        if msg.failed_behavior != "-" and not self.logged_failure:
            failure_info = log_failure(msg)
            if failure_info:
                print(f"[FAILURE] {failure_info}")
            self.logged_failure = True

        # Coordination: wait for previous behavior to complete
        waiting_for_command = msg.behavior_in_progress == "-"

        if self.last_commanded_behavior is not None and not waiting_for_command:
            return
        if self.last_commanded_behavior is not None and waiting_for_command:
            if msg.completed_behavior != self.last_commanded_behavior:
                return

        # Check if mission is complete
        if self.command_index >= len(self.mission_commands):
            print("Demo complete. All behaviors executed successfully.")
            self.demo_complete = True
            return

        # Send next command
        if waiting_for_command or not self.initialized:
            cmd = self.mission_commands[self.command_index]
            print(f"Commanding [{self.command_index + 1}/{len(self.mission_commands)}]: {cmd.behavior_to_execute}")
            self.command_pub.publish(cmd)
            self.last_commanded_behavior = cmd.behavior_to_execute
            self.initialized = True
            self.logged_failure = False
            self.command_index += 1


print("BehaviorCoordinator node defined.")

BehaviorCoordinator node defined.


## 7. Run the Demo

Initialize ROS2, create the coordinator node, and spin. Interrupt the kernel to stop.

In [7]:
rclpy.init()
node = BehaviorCoordinator(MISSION_COMMANDS)

try:
    print("Running explosive breaching demo... (interrupt kernel to stop)")
    rclpy.spin(node)
except KeyboardInterrupt:
    print("\nStopped by user.")
finally:
    node.destroy_node()
    rclpy.shutdown()
    print("Shutdown complete.")

Running explosive breaching demo... (interrupt kernel to stop)

Stopped by user.


RCLError: failed to shutdown: rcl_shutdown already called on the given context, at ./src/rcl/init.c:241

## 8. VLM Queries (Optional)

Use the VLM to analyze images or reason about the scene.

In [8]:
# Text query example
print(vlm_text_query("What safety precautions should be taken during explosive breaching?"))

Explosive breaching is a highly specialized and dangerous operation that requires extensive training, experience, and adherence to strict safety protocols. Here are some key safety precautions that should be followed:

1. **Training and Experience**: Personnel involved in explosive breaching must have extensive training and experience. This includes understanding the principles of explosives, their handling, and the specific techniques required for breaching.

2. **Proper Equipment**: Use only appropriate and certified equipment designed for explosive breaching. This includes protective gear such as helmets, gloves, goggles, and protective suits, as well as specialized tools and explosives.

3. **Site Assessment**: Conduct a thorough site assessment before initiating any explosive breaching operations. Identify potential hazards, assess the structural integrity of the target, and determine the safest entry point.

4. **Communication**: Establish clear communication channels among all t

In [9]:
# Image query example
print(vlm_image_query("vlm/test/room_expo.png", "What objects do you see in this image? List them briefly."))

The image contains the following objects:

1. A blue trash can with a brown lid.
2. A green table with four legs.
3. A pair of white shoes with green accents.
4. A gray couch with a white cushion.

These items are placed on a tiled floor.


In [ ]:
# Image query example
print(vlm_image_query("vlm/test/7447.jpg", "Is the image a push door or pull door"))

Imports loaded.
